# 1 — LHS Geometrical Parameter Generation

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab  
**Purpose:** Generate the geometrical design parameter dataset for the propeller pipeline using Latin Hypercube Sampling (LHS).

This notebook is a self-contained dataset-generation step for the IDEAL Lab propeller pipeline. It defines the design space used by the propeller configurator, generates feasible geometries with Latin Hypercube Sampling, applies geometric and aerodynamic constraints, checks the generated dataset, and exports the final table as a CSV file.


## 1. Imports

Only the libraries required for dataset generation, interpolation, checks, and CSV export are imported here.


In [1]:
# Imports
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.stats import qmc


## 2. Constants

The constants are split into two groups so the notebook is easy to maintain.

### Generator input slider bounds

These values correspond to the user-facing sliders in the propeller generation interface. If the propeller generator changes its slider limits, this is the section that should be updated first. The LHS pipeline then adapts automatically to the same design space.

### Internal pipeline parameters

These are not direct user sliders. They control reproducibility, dataset size, file paths, radial stations, manufacturing limits, aerodynamic assumptions, and interpolation settings.

The inner profile is located inside the hub and is mainly a loft/control profile. Therefore, the aerodynamic constraint is not applied directly at the inner profile radius. Instead, the blade angle is interpolated through the inner, middle, and outer profile angles, and the angle of attack is checked at the hub outer radius. This better represents the smooth lofted blade surface generated in Rhino.


In [2]:
# Configuration


# =============================================================================
# 1) GENERATOR INPUT SLIDER BOUNDS
# =============================================================================
# These values mirror the sliders available in the propeller generation app.
# If the Streamlit/Rhino generator changes, update this dictionary first.


SLIDER_BOUNDS = {
    # General tab
    "radius": (60, 80),
    "ring height": (4, 10),
    "ring thickness": (1, 5),
    "blade count": (3, 6),

    # Inner Profile tab
    "inner thickness": (4, 24),
    "inner max pos": (2, 8),
    "inner camber": (0, 9),
    "inner chord": (5, 11),
    "inner angle": (2, 25),

    # Middle Profile tab
    "mid radial pos": (0.3, 0.7),
    "mid chord": (10, 30),
    "mid angle": (2, 25),

    # Outer Profile tab
    "outer thickness": (4, 24),
    "outer max pos": (2, 8),
    "outer camber": (0, 9),
    "outer chord": (10, 30),
    "outer angle": (2, 25),
}


# Convenience aliases used throughout the notebook.
# They are intentionally derived from SLIDER_BOUNDS so that changing the slider
# dictionary above is enough to update the rest of the pipeline.

RADIUS_MIN_MM, RADIUS_MAX_MM = SLIDER_BOUNDS["radius"]
RING_HEIGHT_MIN_MM, RING_HEIGHT_MAX_MM = SLIDER_BOUNDS["ring height"]
RING_THICKNESS_MIN_MM, RING_THICKNESS_MAX_MM = SLIDER_BOUNDS["ring thickness"]
BLADE_COUNT_MIN, BLADE_COUNT_MAX = SLIDER_BOUNDS["blade count"]

INNER_THICKNESS_MIN_PERCENT, INNER_THICKNESS_MAX_PERCENT = SLIDER_BOUNDS["inner thickness"]
INNER_MAX_POS_MIN, INNER_MAX_POS_MAX = SLIDER_BOUNDS["inner max pos"]
INNER_CAMBER_MIN, INNER_CAMBER_MAX = SLIDER_BOUNDS["inner camber"]
INNER_CHORD_MIN_MM, INNER_CHORD_MAX_MM = SLIDER_BOUNDS["inner chord"]
INNER_ANGLE_MIN_DEG, INNER_ANGLE_MAX_DEG = SLIDER_BOUNDS["inner angle"]

MID_RADIAL_POS_MIN, MID_RADIAL_POS_MAX = SLIDER_BOUNDS["mid radial pos"]
MID_CHORD_MIN_MM, MID_CHORD_MAX_MM = SLIDER_BOUNDS["mid chord"]
MID_ANGLE_MIN_DEG, MID_ANGLE_MAX_DEG = SLIDER_BOUNDS["mid angle"]

OUTER_THICKNESS_MIN_PERCENT, OUTER_THICKNESS_MAX_PERCENT = SLIDER_BOUNDS["outer thickness"]
OUTER_MAX_POS_MIN, OUTER_MAX_POS_MAX = SLIDER_BOUNDS["outer max pos"]
OUTER_CAMBER_MIN, OUTER_CAMBER_MAX = SLIDER_BOUNDS["outer camber"]
OUTER_CHORD_MIN_MM, OUTER_CHORD_MAX_MM = SLIDER_BOUNDS["outer chord"]
OUTER_ANGLE_MIN_DEG, OUTER_ANGLE_MAX_DEG = SLIDER_BOUNDS["outer angle"]


# =============================================================================
# 2) INTERNAL PIPELINE PARAMETERS
# =============================================================================
# These values are not direct input sliders in the generator interface.


# -----------------------------
# Reproducibility
# -----------------------------
GLOBAL_RANDOM_SEED = 99
RADIUS_BLADE_LHS_SEED = 99
GEOMETRY_LHS_SEED = 456

np.random.seed(GLOBAL_RANDOM_SEED)


# -----------------------------
# Dataset size
# -----------------------------
N_SAMPLES = 5000


# -----------------------------
# Output path
# -----------------------------
CSV_DIR = Path("./csv")
OUTPUT_CSV = CSV_DIR / "prop_geometrical_params.csv"


# -----------------------------
# Fixed radial stations
# -----------------------------
# The inner profile is used as a loft/control profile and lies inside the hub.
# The hub outer radius is the first aerodynamically meaningful radius.
INNER_STATION_RADIUS_MM = 4.5
HUB_OUTER_RADIUS_MM = 8.25


# -----------------------------
# Feasibility constraints
# -----------------------------
MIN_ABS_WALL_THICKNESS_MM = 1.0
INNER_SOLIDITY_MAX = 0.7
MID_SOLIDITY_MAX = 0.85

# Internal chord-shaping variable used to generate the outer chord from mid chord.
# It is not a visible slider in the generator app.
TAPER_RATIO_MIN = 0.4
TAPER_RATIO_MAX = 1.0


# -----------------------------
# Aerodynamic angle assumptions
# -----------------------------
RPM = 4000
OMEGA_RAD_PER_S = RPM * 2 * np.pi / 60

V_AXIAL_M_PER_S = 1.0

AOA_MIN_DEG = 0.0
AOA_MAX_DEG = 12.0

# If True, the sampled twist satisfies:
# inner angle >= mid angle >= outer angle
ENFORCE_MONOTONIC_TWIST = True


# -----------------------------
# Interpolation assumptions
# -----------------------------
# Rhino Loft creates a smooth NURBS/spline-like transition between profiles.
# The exact Loft surface is generated inside Rhino, but this notebook uses a
# three-profile cubic spline surrogate for the blade-angle variation along the
# radius. This uses the inner, middle, and outer profile angles rather than only
# the inner and middle profiles.
#
# Change these constants if the CAD interpolation strategy changes.
ANGLE_INTERPOLATION_METHOD = "natural_cubic_spline_three_profiles"
ANGLE_SPLINE_BC_TYPE = "natural"


## 3. Helper functions

These functions keep the generation logic readable and centralize the equations used by the constraints.

The most important helper is the radial angle interpolation. The CAD model is built by lofting blade profiles in Rhino, so the local blade angle between profile stations is treated as a smooth spline-like variation rather than as a purely piecewise-linear quantity. The notebook evaluates this smooth angle distribution at the hub outer radius to constrain the first aerodynamically exposed part of the blade.


In [3]:
def phi_deg(r_mm):
    """Return the inflow angle phi at radius r_mm in degrees."""
    r_m = r_mm / 1000.0
    v_tan = OMEGA_RAD_PER_S * r_m
    return float(np.degrees(np.arctan(V_AXIAL_M_PER_S / v_tan)))


def aoa_pitch_bounds(
    r_mm,
    angle_min_deg,
    angle_max_deg,
    pitch_ceil_deg=None,
):
    """
    Return integer pitch-angle bounds so that the local angle of attack remains
    within the target range, while optionally enforcing monotonic twist.
    """
    phi = phi_deg(r_mm)

    lo = int(np.ceil(phi + AOA_MIN_DEG))
    hi = int(np.floor(phi + AOA_MAX_DEG))

    if pitch_ceil_deg is not None:
        hi = min(hi, int(pitch_ceil_deg))

    lo = max(angle_min_deg, lo)
    hi = min(angle_max_deg, hi)

    if hi < lo:
        hi = lo

    return lo, hi


def scale(u, lo, hi, as_int=True):
    """Scale a uniform LHS value from [0, 1] to the target range [lo, hi]."""
    value = lo + u * (hi - lo)

    if as_int:
        return int(round(value))

    return round(float(value), 3)


def sample_from_integer_candidates(u, candidates):
    """Select one integer value from a candidate array using an LHS sample u."""
    candidates = np.asarray(candidates, dtype=int)

    if len(candidates) == 0:
        raise ValueError("No feasible integer candidates were provided.")

    index = min(int(np.floor(u * len(candidates))), len(candidates) - 1)
    return int(candidates[index])


def interpolate_angle_radially(
    inner_angle_deg,
    mid_angle_deg,
    outer_angle_deg,
    mid_radius_mm,
    outer_radius_mm,
    evaluation_radius_mm,
):
    """
    Interpolate the blade angle along the radius using all three blade profiles.

    The blade geometry is generated by lofting the inner, middle, and outer
    profiles. To approximate the smooth angle variation produced by the loft,
    this function builds a cubic spline through the three profile angles and
    evaluates it at the requested radial location.

    Parameters
    ----------
    inner_angle_deg:
        Angle of the inner profile. This station is inside the hub and acts as
        a loft/control profile.
    mid_angle_deg:
        Angle of the middle profile.
    outer_angle_deg:
        Angle of the outer profile.
    mid_radius_mm:
        Absolute radius of the middle profile in millimetres.
    outer_radius_mm:
        Propeller outer radius in millimetres.
    evaluation_radius_mm:
        Radius where the interpolated angle is needed.
    """
    x = np.array(
        [
            INNER_STATION_RADIUS_MM,
            mid_radius_mm,
            outer_radius_mm,
        ],
        dtype=float,
    )

    y = np.array(
        [
            inner_angle_deg,
            mid_angle_deg,
            outer_angle_deg,
        ],
        dtype=float,
    )

    if not np.all(np.diff(x) > 0):
        raise ValueError(
            "Profile radii must be strictly increasing: "
            "inner station < middle station < outer station."
        )

    if not (x[0] <= evaluation_radius_mm <= x[-1]):
        raise ValueError(
            "The evaluation radius must lie inside the radial profile domain."
        )

    if ANGLE_INTERPOLATION_METHOD == "natural_cubic_spline_three_profiles":
        spline = CubicSpline(x, y, bc_type=ANGLE_SPLINE_BC_TYPE)
        return float(spline(evaluation_radius_mm))

    if ANGLE_INTERPOLATION_METHOD == "linear_three_profiles":
        return float(np.interp(evaluation_radius_mm, x, y))

    raise ValueError(f"Unknown interpolation method: {ANGLE_INTERPOLATION_METHOD}")


def feasible_inner_angles_for_hub_aoa(
    mid_angle_deg,
    outer_angle_deg,
    mid_radius_mm,
    outer_radius_mm,
):
    """
    Return all integer inner angles that make the spline-interpolated angle at
    the hub outer radius satisfy the AoA constraint.

    The inner station itself is not treated as aerodynamically active because
    it lies inside the hub. The first relevant aerodynamic check is performed
    at HUB_OUTER_RADIUS_MM using the interpolated angle from the inner, middle,
    and outer profiles.
    """
    candidate_inner_angles = np.arange(
        INNER_ANGLE_MIN_DEG,
        INNER_ANGLE_MAX_DEG + 1,
    )

    interpolated_hub_angles = np.array([
        interpolate_angle_radially(
            inner_angle_deg=inner_angle,
            mid_angle_deg=mid_angle_deg,
            outer_angle_deg=outer_angle_deg,
            mid_radius_mm=mid_radius_mm,
            outer_radius_mm=outer_radius_mm,
            evaluation_radius_mm=HUB_OUTER_RADIUS_MM,
        )
        for inner_angle in candidate_inner_angles
    ])

    hub_aoa = interpolated_hub_angles - phi_deg(HUB_OUTER_RADIUS_MM)

    feasible_mask = (
        (hub_aoa >= AOA_MIN_DEG)
        & (hub_aoa <= AOA_MAX_DEG)
    )

    if ENFORCE_MONOTONIC_TWIST:
        feasible_mask &= candidate_inner_angles >= mid_angle_deg

    return candidate_inner_angles[feasible_mask]


def select_inner_angle(lhs_value, mid_angle_deg, outer_angle_deg, mid_radius_mm, outer_radius_mm):
    """
    Select the inner angle using the LHS value while enforcing the hub AoA
    constraint after three-profile cubic-spline interpolation.
    """
    feasible_angles = feasible_inner_angles_for_hub_aoa(
        mid_angle_deg=mid_angle_deg,
        outer_angle_deg=outer_angle_deg,
        mid_radius_mm=mid_radius_mm,
        outer_radius_mm=outer_radius_mm,
    )

    if len(feasible_angles) == 0:
        raise ValueError(
            "No feasible inner angle found. Try relaxing the inner angle slider, "
            "the AoA limits, the monotonic twist constraint, or the spline setup."
        )

    return sample_from_integer_candidates(lhs_value, feasible_angles)


## 4. LHS generation

The notebook uses Latin Hypercube Sampling to cover the design space efficiently with a fixed number of configurations. LHS is useful here because it spreads the samples across each variable range more evenly than purely random sampling.

The generation logic follows these steps:

1. Sample the main global variables: propeller radius and blade count.
2. Sample the profile-shape variables for the inner, middle, and outer blade profiles.
3. Apply feasibility constraints while generating each configuration:
   - **Solidity limit:** avoids unrealistically crowded blades by limiting the ratio between blade chord area and annular circumference.
   - **Minimum printable wall thickness:** avoids profiles that would be too thin to manufacture reliably.
   - **Angle-of-attack range:** keeps the local blade pitch within a practical aerodynamic range.
   - **Monotonic twist:** enforces a decreasing pitch from root to tip when enabled.
4. Compute the inner angle last. Since the inner profile lies inside the hub, the constraint is applied at the hub outer radius using a cubic spline through the inner, middle, and outer profile angles.
5. Export the final set of geometrical parameters in the exact column order required by the pipeline.


In [4]:
def generate_lhs_geometrical_parameters(n_samples):
    """Generate propeller geometrical parameters using LHS."""

    # Radius and blade count are sampled independently.
    radius_blade_samples = qmc.LatinHypercube(
        d=2,
        seed=RADIUS_BLADE_LHS_SEED,
    ).random(n=n_samples)

    radii = np.round(
        qmc.scale(
            radius_blade_samples[:, 0:1],
            [RADIUS_MIN_MM],
            [RADIUS_MAX_MM],
        )
    ).astype(int).flatten()

    blade_counts = np.round(
        qmc.scale(
            radius_blade_samples[:, 1:2],
            [BLADE_COUNT_MIN],
            [BLADE_COUNT_MAX],
        )
    ).astype(int).flatten()

    # Remaining LHS variables used by the geometrical generator.
    # The current default uses 13 variables. If ring height/thickness are later
    # sampled directly, two extra LHS dimensions are enabled automatically.
    geometry_lhs_dimension = 13  # ring height/thickness are computed deterministically, not sampled
    lhs_samples = qmc.LatinHypercube(
        d=geometry_lhs_dimension,
        seed=GEOMETRY_LHS_SEED,
    ).random(n=n_samples)

    records = []

    for config_id in range(n_samples):
        radius = int(radii[config_id])
        blade_count = int(blade_counts[config_id])

        # Solidity-dependent chord upper bound at the first aerodynamically
        # relevant radius, which is the hub outer radius.
        inner_chord_max = min(
            int(INNER_SOLIDITY_MAX * 2 * np.pi * HUB_OUTER_RADIUS_MM / blade_count),
            INNER_CHORD_MAX_MM,
        )

        # Inner station geometry except the angle.
        # The inner angle is selected later because its feasible values depend
        # on the middle and outer profile angles through the cubic spline.
        inner_chord = scale(
            lhs_samples[config_id, 3],
            INNER_CHORD_MIN_MM,
            inner_chord_max,
        )

        inner_thickness_min = max(
            INNER_THICKNESS_MIN_PERCENT,
            int(np.ceil(MIN_ABS_WALL_THICKNESS_MM * 100 / inner_chord)),
        )

        inner_thickness = scale(
            lhs_samples[config_id, 0],
            inner_thickness_min,
            INNER_THICKNESS_MAX_PERCENT,
        )

        inner_max_pos = scale(
            lhs_samples[config_id, 1],
            INNER_MAX_POS_MIN,
            INNER_MAX_POS_MAX,
        )

        inner_camber = scale(
            lhs_samples[config_id, 2],
            INNER_CAMBER_MIN,
            INNER_CAMBER_MAX,
        )

        # Middle station geometry and pitch.
        mid_radial_pos = round(scale(
            lhs_samples[config_id, 5],
            MID_RADIAL_POS_MIN,
            MID_RADIAL_POS_MAX,
            as_int=False,
        ), 1)

        mid_radius = mid_radial_pos * radius

        mid_chord_hi = min(
            int(MID_SOLIDITY_MAX * 2 * np.pi * mid_radius / blade_count),
            MID_CHORD_MAX_MM,
        )

        mid_chord = scale(
            lhs_samples[config_id, 6],
            MID_CHORD_MIN_MM,
            mid_chord_hi,
        )

        mid_angle_lo, mid_angle_hi = aoa_pitch_bounds(
            r_mm=mid_radius,
            angle_min_deg=MID_ANGLE_MIN_DEG,
            angle_max_deg=MID_ANGLE_MAX_DEG,
        )

        mid_angle = scale(
            lhs_samples[config_id, 7],
            mid_angle_lo,
            mid_angle_hi,
        )

        # Outer station geometry.
        outer_chord_hi = max(
            OUTER_CHORD_MIN_MM,
            min(mid_chord, OUTER_CHORD_MAX_MM),
        )

        outer_chord = scale(
            lhs_samples[config_id, 8],
            OUTER_CHORD_MIN_MM,
            outer_chord_hi,
        )

        outer_thickness_min = max(
            OUTER_THICKNESS_MIN_PERCENT,
            int(np.ceil(MIN_ABS_WALL_THICKNESS_MM * 100 / outer_chord)),
        )

        outer_thickness = scale(
            lhs_samples[config_id, 9],
            outer_thickness_min,
            min(inner_thickness, OUTER_THICKNESS_MAX_PERCENT),
        )

        outer_max_pos = scale(
            lhs_samples[config_id, 11],
            OUTER_MAX_POS_MIN,
            OUTER_MAX_POS_MAX,
        )

        outer_camber = scale(
            lhs_samples[config_id, 10],
            OUTER_CAMBER_MIN,
            OUTER_CAMBER_MAX,
        )

        outer_angle_lo, outer_angle_hi = aoa_pitch_bounds(
            r_mm=radius,
            angle_min_deg=OUTER_ANGLE_MIN_DEG,
            angle_max_deg=OUTER_ANGLE_MAX_DEG,
            pitch_ceil_deg=mid_angle if ENFORCE_MONOTONIC_TWIST else None,
        )

        outer_angle = scale(
            lhs_samples[config_id, 12],
            outer_angle_lo,
            outer_angle_hi,
        )

        # Inner angle logic.
        # The effective root AoA is evaluated at the hub outer radius using a
        # cubic spline through the inner, middle, and outer profile angles.
        inner_angle = select_inner_angle(
            lhs_value=lhs_samples[config_id, 4],
            mid_angle_deg=mid_angle,
            outer_angle_deg=outer_angle,
            mid_radius_mm=mid_radius,
            outer_radius_mm=radius,
        )

        # Shroud ring geometry.
        # The ring height is set to the minimum required to fully cover the
        # outer airfoil of this specific configuration. This removes the ring
        # as a free aerodynamic variable while keeping it as tight as possible
        # around each blade.
        #
        # The thickness is set to exactly half the height.
        outer_angle_rad = np.radians(outer_angle)

        minimum_required_ring_height = (
            outer_chord * np.sin(outer_angle_rad)
            + outer_thickness * np.sin(outer_angle_rad) / 6
            + outer_camber * np.sin(outer_angle_rad) / 3
        )

        ring_height = float(np.clip(
            np.ceil(minimum_required_ring_height + 1),
            RING_HEIGHT_MIN_MM,
            RING_HEIGHT_MAX_MM,
        ))

        ring_thickness = float(np.clip(
            ring_height / 2,
            RING_THICKNESS_MIN_MM,
            RING_THICKNESS_MAX_MM,
        ))

        records.append({
            "config_id": config_id,
            "radius": radius,
            "ring height": ring_height,
            "ring thickness": ring_thickness,
            "blade count": blade_count,
            "inner thickness": inner_thickness,
            "inner max pos": inner_max_pos,
            "inner camber": inner_camber,
            "inner chord": inner_chord,
            "inner angle": inner_angle,
            "mid radial pos": mid_radial_pos,
            "mid chord": mid_chord,
            "mid angle": mid_angle,
            "outer thickness": outer_thickness,
            "outer max pos": outer_max_pos,
            "outer camber": outer_camber,
            "outer chord": outer_chord,
            "outer angle": outer_angle,
        })

    columns = [
        "config_id",
        "radius",
        "ring height",
        "ring thickness",
        "blade count",
        "inner thickness",
        "inner max pos",
        "inner camber",
        "inner chord",
        "inner angle",
        "mid radial pos",
        "mid chord",
        "mid angle",
        "outer thickness",
        "outer max pos",
        "outer camber",
        "outer chord",
        "outer angle",
    ]

    return pd.DataFrame(records, columns=columns)


## 5. Generate and export the dataset

The CSV directory is created automatically if it does not already exist.  
The final dataset is exported to:

```text
./csv/prop_geometrical_params.csv
```


In [5]:
# Generate LHS dataset
geometrical_params_df = generate_lhs_geometrical_parameters(N_SAMPLES)

# Ensure output directory exists
CSV_DIR.mkdir(parents=True, exist_ok=True)

# Export CSV
geometrical_params_df.to_csv(OUTPUT_CSV, index=False)

print(f"Generated {len(geometrical_params_df)} configurations.")
print(f"CSV saved to: {OUTPUT_CSV}")
print(f"Columns: {list(geometrical_params_df.columns)}")

geometrical_params_df.head()


Generated 5000 configurations.
CSV saved to: csv\prop_geometrical_params.csv
Columns: ['config_id', 'radius', 'ring height', 'ring thickness', 'blade count', 'inner thickness', 'inner max pos', 'inner camber', 'inner chord', 'inner angle', 'mid radial pos', 'mid chord', 'mid angle', 'outer thickness', 'outer max pos', 'outer camber', 'outer chord', 'outer angle']


,config_id,radius,ring height,ring thickness,blade count,inner thickness,inner max pos,inner camber,inner chord,inner angle,mid radial pos,mid chord,mid angle,outer thickness,outer max pos,outer camber,outer chord,outer angle
0,0,67,5,2,5,17,6,4,7,23,0.5,22,9,12,7,4,12,3
1,1,60,4,1,3,19,5,6,11,25,0.6,17,11,14,5,8,16,8
2,2,80,10,4,5,18,5,5,6,17,0.4,23,15,11,4,2,18,10
3,3,66,9,1,6,22,4,7,5,22,0.5,12,7,13,2,2,10,7
4,4,67,9,2,5,18,4,8,6,22,0.4,21,15,17,5,3,15,8


## 6. Minimal consistency check

This cell verifies that the exported table contains exactly the requested columns in the required order.


In [6]:
expected_columns = [
    "config_id",
    "radius",
    "ring height",
    "ring thickness",
    "blade count",
    "inner thickness",
    "inner max pos",
    "inner camber",
    "inner chord",
    "inner angle",
    "mid radial pos",
    "mid chord",
    "mid angle",
    "outer thickness",
    "outer max pos",
    "outer camber",
    "outer chord",
    "outer angle",
]

assert list(geometrical_params_df.columns) == expected_columns, "Column order does not match the required specification."
assert geometrical_params_df.shape[1] == len(expected_columns), "Unexpected number of columns."

print("Column order and number of columns are correct.")


Column order and number of columns are correct.


## 7. Dataset consistency checks

This final section verifies that the generated geometries are valid before the CSV is used downstream.

The checks cover:

- **Generator slider bounds:** every exported value must remain inside the limits available in the propeller configurator.
- **Solidity:** the blade chord must not occupy too much of the local annular circumference. This prevents very crowded blade geometries that are difficult to interpret aerodynamically and may be unsuitable for the configurator.
- **Minimum wall thickness:** the absolute profile thickness must remain above the selected manufacturing threshold.
- **Monotonic twist:** when enabled, the pitch angle should decrease from the inner profile to the middle profile and then to the outer profile.
- **Angle of attack:** the local pitch angle minus the inflow angle must stay inside the selected aerodynamic range. For the hub region, the pitch angle is evaluated by cubic-spline interpolation through the inner, middle, and outer profile angles at the hub outer radius.


In [7]:
# Dataset consistency checks

all_ok = True


def check(name, series, lo, hi=None, invert=False):
    """Print a compact pass/fail check for a numerical series."""
    global all_ok

    vmin = round(float(series.min()), 3)
    vmax = round(float(series.max()), 3)

    if invert:
        ok = (series >= lo).all()
        expected = f">= {lo}"
    else:
        ok = ((series >= lo).all() and (series <= hi).all())
        expected = f"[{lo}, {hi}]"

    status = "  OK  " if ok else " FAIL "
    print(f"  [{status}]  {name:<32} actual [{vmin}, {vmax}]  expected {expected}")

    if not ok:
        all_ok = False


df_check = geometrical_params_df.rename(
    columns={
        "blade count": "blade_count",
        "inner chord": "inner_chord",
        "inner angle": "inner_angle",
        "inner thickness": "inner_thickness",
        "inner camber": "inner_camber",
        "inner max pos": "inner_max_position",
        "mid radial pos": "mid_radial_pos",
        "mid chord": "mid_chord",
        "mid angle": "mid_angle",
        "outer chord": "outer_chord",
        "outer angle": "outer_angle",
        "outer thickness": "outer_thickness",
        "outer camber": "outer_camber",
        "outer max pos": "outer_max_position",
    }
)


print("Generator slider bounds")
print("-" * 76)

check("radius", df_check["radius"], RADIUS_MIN_MM, RADIUS_MAX_MM)
check("ring_height", geometrical_params_df["ring height"], RING_HEIGHT_MIN_MM, RING_HEIGHT_MAX_MM)
check("ring_thickness", geometrical_params_df["ring thickness"], RING_THICKNESS_MIN_MM, RING_THICKNESS_MAX_MM)
check("blade_count", df_check["blade_count"], BLADE_COUNT_MIN, BLADE_COUNT_MAX)

check("inner_chord", df_check["inner_chord"], INNER_CHORD_MIN_MM, INNER_CHORD_MAX_MM)
check("inner_angle", df_check["inner_angle"], INNER_ANGLE_MIN_DEG, INNER_ANGLE_MAX_DEG)
check("inner_thickness", df_check["inner_thickness"], INNER_THICKNESS_MIN_PERCENT, INNER_THICKNESS_MAX_PERCENT)
check("inner_camber", df_check["inner_camber"], INNER_CAMBER_MIN, INNER_CAMBER_MAX)
check("inner_max_position", df_check["inner_max_position"], INNER_MAX_POS_MIN, INNER_MAX_POS_MAX)

check("mid_radial_pos", df_check["mid_radial_pos"], MID_RADIAL_POS_MIN, MID_RADIAL_POS_MAX)
check("mid_chord", df_check["mid_chord"], MID_CHORD_MIN_MM, MID_CHORD_MAX_MM)
check("mid_angle", df_check["mid_angle"], MID_ANGLE_MIN_DEG, MID_ANGLE_MAX_DEG)

check("outer_chord", df_check["outer_chord"], OUTER_CHORD_MIN_MM, OUTER_CHORD_MAX_MM)
check("outer_angle", df_check["outer_angle"], OUTER_ANGLE_MIN_DEG, OUTER_ANGLE_MAX_DEG)
check("outer_thickness", df_check["outer_thickness"], OUTER_THICKNESS_MIN_PERCENT, OUTER_THICKNESS_MAX_PERCENT)
check("outer_camber", df_check["outer_camber"], OUTER_CAMBER_MIN, OUTER_CAMBER_MAX)
check("outer_max_position", df_check["outer_max_position"], OUTER_MAX_POS_MIN, OUTER_MAX_POS_MAX)


print()
print("Solidity")
print("-" * 76)

solidity_inner = (
    df_check["inner_chord"] * df_check["blade_count"]
) / (
    2 * np.pi * HUB_OUTER_RADIUS_MM
)

solidity_mid = (
    df_check["mid_chord"] * df_check["blade_count"]
) / (
    2 * np.pi * df_check["radius"] * df_check["mid_radial_pos"]
)

check("inner solidity at hub radius", solidity_inner, 0, INNER_SOLIDITY_MAX)
check("mid solidity", solidity_mid, 0, MID_SOLIDITY_MAX)


print()
print("Printability  (abs wall thickness >= minimum)")
print("-" * 76)

inner_abs_thickness = (
    df_check["inner_chord"] * df_check["inner_thickness"] / 100
)

outer_abs_thickness = (
    df_check["outer_chord"] * df_check["outer_thickness"] / 100
)

mid_thickness = (
    df_check["inner_thickness"]
    + df_check["mid_radial_pos"]
    * (df_check["outer_thickness"] - df_check["inner_thickness"])
)

mid_abs_thickness = (
    df_check["mid_chord"] * mid_thickness / 100
)

check("inner abs thickness [mm]", inner_abs_thickness, MIN_ABS_WALL_THICKNESS_MM, invert=True)
check("mid abs thickness [mm]", mid_abs_thickness, MIN_ABS_WALL_THICKNESS_MM, invert=True)
check("outer abs thickness [mm]", outer_abs_thickness, MIN_ABS_WALL_THICKNESS_MM, invert=True)


print()
print("Twist monotonicity")
print("-" * 76)

twist_monotonic = (
    (df_check["inner_angle"] >= df_check["mid_angle"])
    & (df_check["mid_angle"] >= df_check["outer_angle"])
)

twist_status = "  OK  " if twist_monotonic.all() else " FAIL "
print(
    f"  [{twist_status}]  "
    f"{twist_monotonic.sum()}/{len(df_check)} configs have monotonic twist"
)

if not twist_monotonic.all():
    all_ok = False


print()
print(f"AoA range check  (target: {AOA_MIN_DEG:.0f}° – {AOA_MAX_DEG:.0f}° at {RPM} RPM)")
print("-" * 76)


def inflow_angle_deg(r_mm_series):
    """Compute inflow angle phi [deg] at each station."""
    r_m = r_mm_series / 1000
    v_tan = OMEGA_RAD_PER_S * r_m
    return np.degrees(np.arctan(V_AXIAL_M_PER_S / v_tan))


# Inner/hub effective station.
mid_radius_series = df_check["mid_radial_pos"] * df_check["radius"]

hub_interpolated_angle = np.array([
    interpolate_angle_radially(
        inner_angle_deg=inner_angle,
        mid_angle_deg=mid_angle,
        outer_angle_deg=outer_angle,
        mid_radius_mm=mid_radius,
        outer_radius_mm=outer_radius,
        evaluation_radius_mm=HUB_OUTER_RADIUS_MM,
    )
    for inner_angle, mid_angle, outer_angle, mid_radius, outer_radius in zip(
        df_check["inner_angle"],
        df_check["mid_angle"],
        df_check["outer_angle"],
        mid_radius_series,
        df_check["radius"],
    )
])

aoa_hub = hub_interpolated_angle - phi_deg(HUB_OUTER_RADIUS_MM)

n_ok_hub = ((aoa_hub >= AOA_MIN_DEG) & (aoa_hub <= AOA_MAX_DEG)).sum()
n_hi_hub = (aoa_hub > AOA_MAX_DEG).sum()
n_lo_hub = (aoa_hub < AOA_MIN_DEG).sum()
pct_hub = round(100 * n_ok_hub / len(aoa_hub))

status_hub = "  OK  " if n_hi_hub == 0 and n_lo_hub == 0 else " FAIL "

print(
    f"  [{status_hub}]  hub    AoA  "
    f"[{aoa_hub.min():.1f}°, {aoa_hub.max():.1f}°]  "
    f"in [{AOA_MIN_DEG:.0f}°,{AOA_MAX_DEG:.0f}°]: {n_ok_hub}/{len(aoa_hub)} ({pct_hub}%)  "
    f"|  >{AOA_MAX_DEG:.0f}°: {n_hi_hub}  |  <{AOA_MIN_DEG:.0f}°: {n_lo_hub}"
)

if n_hi_hub != 0 or n_lo_hub != 0:
    all_ok = False


# Mid station.
phi_mid = inflow_angle_deg(mid_radius_series)
aoa_mid = df_check["mid_angle"] - phi_mid

n_ok_mid = ((aoa_mid >= AOA_MIN_DEG) & (aoa_mid <= AOA_MAX_DEG)).sum()
n_hi_mid = (aoa_mid > AOA_MAX_DEG).sum()
n_lo_mid = (aoa_mid < AOA_MIN_DEG).sum()
pct_mid = round(100 * n_ok_mid / len(aoa_mid))

status_mid = "  OK  " if n_hi_mid == 0 and n_lo_mid == 0 else " FAIL "

print(
    f"  [{status_mid}]  mid    AoA  "
    f"[{aoa_mid.min():.1f}°, {aoa_mid.max():.1f}°]  "
    f"in [{AOA_MIN_DEG:.0f}°,{AOA_MAX_DEG:.0f}°]: {n_ok_mid}/{len(aoa_mid)} ({pct_mid}%)  "
    f"|  >{AOA_MAX_DEG:.0f}°: {n_hi_mid}  |  <{AOA_MIN_DEG:.0f}°: {n_lo_mid}"
)

if n_hi_mid != 0 or n_lo_mid != 0:
    all_ok = False


# Outer station.
phi_outer = inflow_angle_deg(df_check["radius"])
aoa_outer = df_check["outer_angle"] - phi_outer

n_ok_outer = ((aoa_outer >= AOA_MIN_DEG) & (aoa_outer <= AOA_MAX_DEG)).sum()
n_hi_outer = (aoa_outer > AOA_MAX_DEG).sum()
n_lo_outer = (aoa_outer < AOA_MIN_DEG).sum()
pct_outer = round(100 * n_ok_outer / len(aoa_outer))

status_outer = "  OK  " if n_hi_outer == 0 and n_lo_outer == 0 else " FAIL "

print(
    f"  [{status_outer}]  outer  AoA  "
    f"[{aoa_outer.min():.1f}°, {aoa_outer.max():.1f}°]  "
    f"in [{AOA_MIN_DEG:.0f}°,{AOA_MAX_DEG:.0f}°]: {n_ok_outer}/{len(aoa_outer)} ({pct_outer}%)  "
    f"|  >{AOA_MAX_DEG:.0f}°: {n_hi_outer}  |  <{AOA_MIN_DEG:.0f}°: {n_lo_outer}"
)

if n_hi_outer != 0 or n_lo_outer != 0:
    all_ok = False


print()
print("=" * 76)
print("ALL CHECKS PASSED ✓" if all_ok else "SOME CHECKS FAILED")
print("=" * 76)


Generator slider bounds
----------------------------------------------------------------------------
  [  OK  ]  radius                           actual [60.0, 80.0]  expected [60, 80]
  [  OK  ]  ring_height                      actual [4.0, 10.0]  expected [4, 10]
  [  OK  ]  ring_thickness                   actual [1.0, 5.0]  expected [1, 5]
  [  OK  ]  blade_count                      actual [3.0, 6.0]  expected [3, 6]
  [  OK  ]  inner_chord                      actual [5.0, 11.0]  expected [5, 11]
  [  OK  ]  inner_angle                      actual [16.0, 25.0]  expected [2, 25]
  [  OK  ]  inner_thickness                  actual [10.0, 24.0]  expected [4, 24]
  [  OK  ]  inner_camber                     actual [0.0, 9.0]  expected [0, 9]
  [  OK  ]  inner_max_position               actual [2.0, 8.0]  expected [2, 8]
  [  OK  ]  mid_radial_pos                   actual [0.3, 0.7]  expected [0.3, 0.7]
  [  OK  ]  mid_chord                        actual [10.0, 30.0]  expected [10, 3